# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'https://raw.githubusercontent.com/jasontpv/DS2002FA26/refs/heads/main/notebooks/06-midterm-walmart/data/product_catalog.csv' #I changed the data to the link from GitHub since it was having trouble finding it
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
poptart_products = cat[cat['product_name'].str.lower().str.contains('pop', na=False)] #looks for products that have a pop in their name, therefore indicating it's a poptart

print("Number of rows that look like Pop-Tart products:", len(poptart_products)) #prints how many products look like pop-tarts
print(poptart_products)

Number of rows that look like Pop-Tart products: 4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


**Markdown:** In this question we made all the product names the same format of lower case so that we could then look for commonalities like "pop" in the name, which would indicate the potential for the item to be a pop-tart. This is important because we are specifically trying to analyze poptart sales, so the first step is isolating those from the rest of the products.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
straw_rows = poptart_products[poptart_products['product_name'].str.lower().str.contains('strawberry',na=False)] #all the items that are pop-tart products and contains strawberry in the product name
print('Strawberry SKUs and names:')
print(straw_rows[['sku','product_name']]) #printing the strawberry poptart names and skus
other_pop = poptart_products[poptart_products['product_name'].str.lower().str.contains('strawberry',na=False)==False] #poptart items that don't contain strawberry
print('Not strawberry rows in poptarts:')
print(other_pop) #print it which tells you what the other poptarts are

Strawberry SKUs and names:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry
Not strawberry rows in poptarts:
Empty DataFrame
Columns: [sku, product_name, category, subcategory, unit_cost, supplier, package_size]
Index: []


**Pop-Tart rows that are a different product:** As you can see by the empty dataframe, there are no non-strawberry rows in this data set. However, should there be non-strawberry poptarts in another data set, they would stay separate from strawberry poptarts with the code we provided for other_pop. This is important because they are not the same product and need to be treated as separate.

**Markdown:** Here we split up poptarts into two categories, strawberry and non-strawberry. This is important because we are not looking for any trend in poptarts, but instead specifically in strawberry poptarts. It is also good to print the other_pop (non-strawberry table) out, just in case something like strawberries was not counted, and just to see what other flavors of poptarts we might want to look at selling.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
orginal_rows = len(cat)
sku_map = {'PT-12': 'POPTART-STRAW', 'SKU#459812':'POPTART-STRAW','POPTART-STRAW':'POPTART-STRAW','PT_STRAWBERRY':'POPTART-STRAW'} #changing all the various strawberry poptart skus into one
cat = cat.assign(canonical = cat['sku'].map(sku_map).fillna(cat['sku'])) #fillna with the normal sku makes sure that we aren't turning non-matching skus into NaN
print(cat[['sku','product_name','canonical']]) #which columns we want to print

              sku            product_name      canonical
0           PT-12    pop-tarts strawberry  POPTART-STRAW
1      SKU#459812     Strawberry Pop-Tart  POPTART-STRAW
2   POPTART-STRAW    Pop-Tarts Strawberry  POPTART-STRAW
3   PT_STRAWBERRY    pop-tarts strawberry  POPTART-STRAW
4          WTR-24      BOTTLED WATER 24PK         WTR-24
5       WATER24PK   Bottled Water 24-Pack      WATER24PK
6        WTR_24PK         Water - 24 Pack       WTR_24PK
7        BATT-100        BATTERIES AA 4PK       BATT-100
8         BAT-AA4     AA Batteries 4-Pack        BAT-AA4
9     BATT_AA_4PK     AA Batteries 4-Pack    BATT_AA_4PK
10       FLASH-01          FLASHLIGHT LED       FLASH-01
11      LED-FLASH          FLASHLIGHT LED      LED-FLASH
12        BEER-24    Light Beer (24-pack)        BEER-24
13       LTBEER24    Light Beer (24-pack)       LTBEER24
14        BG-MONO     Monopoly Board Game        BG-MONO
15    MONOPOLY-01     Monopoly Board Game    MONOPOLY-01
16      GAME-MONO   Board Game 

**Markdown:** Here we consolidated skus so that the same product wouldn't show up as multiple different products in the system. This is importnat because then it is easier to analyze data instead of one item being represented by many different skus and then making it hard to determine what we should be looking at.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print('Before normalizing:')
print(cat['category'].value_counts(dropna=False)) #showing distinct values before with the counts
print('Distinct categories before:')
print(cat['category'].unique())
cat['normalized'] = cat['category'].str.strip().str.title() #strip and title case
cat['normalized'] = cat['normalized'].replace({'Breakfast Foods': 'Breakfast','Emergency Supplies':'Emergency','Alcoholic Beverages':'Alcohol','Groceries':'Grocery','Baby':'Baby Products'}) #making it so that any pair that is two labels for one thing are put in the same category
print('After normalizing:')
print(cat['normalized'].value_counts(dropna=False)) #showing distinct values after with the counts
print('Distinct categories after:')
print(cat['normalized'].unique())

Before normalizing:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
Distinct categories before:
['Breakfast Foods' 'Breakfast' 'breakfast' 'Emergency Supplies'
 'Emergency' 'EMERGENCY' 'Alcoholic Beverages' 'ALCOHOL' 'ENTERTAINMENT'
 'entertainment' 'Electronics' 'electronics' 'Groceries' 'GROCERY'
 'Grocery' 'grocery' 'health' 'ELECTRO

**Markdown:** Here we grouped items into the same categories so that it would be easier for us to see what our main categories were and therefore which items are most popular. This is important because it makes the data cleaner to view and also helps us know where the majority of the items we are selling come from which can help in later analysis.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
cat['normal'] = (cat['product_name'].str.lower()
                         .str.replace(r'[^a-z ]', ' ', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w)))) #lowercase, strip punctuation, and sorted the words
candidates = cat.groupby('normal')['sku'].agg(list).loc[lambda s: s.str.len()>1] #grouped by the normalization from above, and then the len>1 puts the groups holding more than one SKU
candidates

,sku
normal,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
aid first kit,"[FAID-01, FIRSTAID-KIT]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"


**Markdown:** Here we made a list that shows which products have multiple representations. This is valueable because then it means we understand where items are the same but just may have registered differently across purchase. Especially for a case like this one that revolves around inventory, it is important to understand which good you are looking at and exact quantities, we don't want to accidentally separate the same item out across the data because we didn't realize they were the same.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
assert cat['canonical'].notna().all() #no nulls
assert len(cat) == orginal_rows #same number of rows before and after
print('All is good!')

All is good!


**Markdown:** Here we just proved the code did not delete any data and that it did what we wanted it to do. This is important because if something goes wrong early on in this data-cleaning stage, it will greatly impact our analysis later, and we don't want it to be for the worse.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**Answer a)** Another product family that seems fragmented is bottled water. This is because not only does it have 3 different skus for the same product, but also inconsistency in naming means it does not appear on our candidate list. The program can't tell that PK and pack are the same thing or that water and bottled water are the same. To group them then, we would have to make this distinction in our code that we want to equate PK and pack as well as water with bottled water so that our data runs smooth and shows all the info without fragmentations. Another product family that is also fragmented and in a similar vein to water bottles is batteries. There are 3 different skus associated with the batteries, but only 2 show in our candidate list because python again runs into the issue of pack vs PK. So clarifying that the word and abbreviation mean the same thing is how we would then know to group them all together.

**Answer b)** Consolidation becomes a business decision when two items within one category could have very different values or worths, or when it is not clear if the items are truly interchangeable. For example, in electronics, should we combine Roku Streaming Stick and Streaming Device? The Roku stick is a small stick that is plugged into an existing device whereas a streaming device could be buying a whole new device like a TV. These are different items and perform similar, but not exactly the same function. I would probably check with the inventory manager to clear this up and determine categories. This is because grouping too broadly could cause wrong analysis making data like inventory or sales charts wrong. An inventory manager would be able to confirm if they were the same item or, if not, which was the true best seller. The same could also be said for the portable charger and portable phone charger. If this data is really from a massive natural disaster, a portable charger could be an electric generator and not just a regular phone charger.